# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library, with extensive use of entity `@id` references for fields, record sets, and columns throughout the workflow.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Record sets, fields, and columns are uniquely identified by their `@id`. This ensures consistent referencing and manipulation throughout your workflow.

In [ ]:
# List available record sets and their @id
record_sets = list(dataset.record_sets())
print("Record Sets found:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '<no name>')}")

# For each record set, show fields and columns
for rs in record_sets:
    print(f"\nFields in Record Set {rs['@id']}:")
    fields = dataset.fields(record_set=rs['@id'])
    for f in fields:
        fname = f.get('name', '<no name>')
        print(f"  Field @id: {f['@id']} | name: {fname}")
        columns = f.get('columns', [])
        for col in columns:
            colid = col.get('@id', 'unknown')
            print(f"    Column @id: {colid}")

# For demonstration, print example records from the first detected record set
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"\nExample records for Record Set {first_record_set_id}:")
    count = 0
    for x in dataset.records(record_set=first_record_set_id):
        print(x)
        count += 1
        if count >= 3:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference record set `@id`s and field `@id`s as found above.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns from the main record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns for record set (by @id): {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing steps: filter, normalize, and group fields.

**All steps reference fields and columns using their `@id`.**

In [ ]:
# Example: Select the main record set and a numeric field (column) via their @id
# Find a numeric column @id (e.g., 'cr:Age') for illustration
main_df = dataframes.get(main_record_set_id, pd.DataFrame())
numeric_field_id = None
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback to generic column
    numeric_field_id = main_df.columns[0] if not main_df.empty else None

threshold = 50
if numeric_field_id:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by a key categorical column (e.g., 'cr:Sex')
    group_field_id = None
    for col in filtered_df.columns:
        if 'sex' in col.lower() or 'gender' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped (mean {numeric_field_id}) by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using referenced `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field if present
if numeric_field_id and not main_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If group_field_id is present, plot comparison
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id} (@id)")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Through this notebook, you have:
- Loaded and explored the FAIR^2 dataset using the Croissant schema via `mlcroissant`.
- Referenced all entities by their `@id` for robust, reproducible analysis.
- Performed initial filtering, normalization, grouping, and visualized numeric distributions.

For further analysis, reference additional fields and record sets using their `@id`s as identified above, and extend EDA or modeling as needed.